# 🚀 Getting Started with TorchTPU: Training a CNN from Scratch on Cloud TPU v5e

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/12U4zAy7-tiZy723WzkQlooMQqRN7kAGK?resourcekey=0-RgZjpP473V1kbC60am2i3Q)

Welcome to the hands-on lab of the **TorchTPU DevLabs Training Workshop**!

Before training large foundation models or fine-tuning with LoRA, it is useful to understand how PyTorch executes directly on Cloud TPUs with a clean, lightweight model.

### In this notebook, you will:
1. **Initialize the TPU**: Target Cloud TPU with standard PyTorch syntax: `device = torch.device('tpu')`.
2. **Load an Open Image Dataset**: Use **Oxford-IIIT Pet** (or CIFAR-10) directly via `torchvision.datasets` without manual download steps or API keys.
3. **Build a Standard CNN**: Define a modular Convolutional Neural Network (`nn.Module`) using Conv2d, BatchNorm, ReLU, and Linear layers.
4. **Learn Core TPU Invariants**:
   - **Batch Dimensions**: Fixed batch shapes (`drop_last=True`) ensure single-pass compilation.
   - **Compute Precision**: Training in `bfloat16` to leverage TPU Matrix Multiply Units (MXUs).
   - **Asynchronous Execution**: Decoupling loss metrics from host CPU synchronization.
5. **Compile with `torch.compile`**: Compare first-step graph compilation against cached steady-state execution.
6. **Evaluate & Visualize**: Run validation on test images and display predictions.

---
## Step 1: Environment Verification

In an environment equipped with Cloud TPU support, PyTorch recognizes the TPU accelerator directly via standard device APIs.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import time
import matplotlib.pyplot as plt

print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")

# =====================================================================
# TPU Initialization:
# TorchTPU registers the 'tpu' device directly with PyTorch's dispatcher.
# =====================================================================
device = torch.device("tpu")
print(f"Selected device: {device}")

# Quick sanity check: tensor allocation and computation on TPU
x = torch.randn(512, 512, dtype=torch.bfloat16, device=device)
y = torch.matmul(x, x)
print(f"Sanity check passed: {y.shape} on {y.device} ({y.dtype})")


---
## Step 2: Load Dataset (Oxford-IIIT Pet)

We use the **Oxford-IIIT Pet** dataset (37 breeds of dogs and cats). It downloads automatically via `torchvision` and offers a fun, recognizable multiclass classification task.

If running offline or with bandwidth limits, CIFAR-10 can be substituted with the exact same data pipeline.

In [ ]:
# Standard image transformations
# Images are resized to 224x224 so every batch has static shape [64, 3, 224, 224]
IMAGE_SIZE = 224
BATCH_SIZE = 64

transform_train = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transform_val = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("Loading Oxford-IIIT Pet dataset...")
try:
    train_dataset = torchvision.datasets.OxfordIIITPet(root="./data", split="trainval", download=True, transform=transform_train)
    val_dataset = torchvision.datasets.OxfordIIITPet(root="./data", split="test", download=True, transform=transform_val)
    NUM_CLASSES = 37
    classes = train_dataset.classes
except Exception as e:
    print(f"Falling back to CIFAR-10: {e}")
    train_dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)
    val_dataset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_val)
    NUM_CLASSES = 10
    classes = train_dataset.classes

print(f"Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}, Classes: {NUM_CLASSES}")

# =====================================================================
# TPU Best Practice:
# 1. Use drop_last=True so every batch has identical shape [64, 3, 224, 224],
#    preventing recompilation on a ragged final batch.
# 2. Use num_workers=0 (or multiprocessing_context='spawn') after TPU init to
#    avoid POSIX fork() deadlocks in multi-threaded PJRT processes.
# 3. Seeded shuffle on val_loader mixes all 37 breeds per batch (Oxford-IIIT Pet
#    is sorted by breed on disk) for balanced evaluation and diverse plots.
# =====================================================================
val_gen = torch.Generator().manual_seed(42)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=val_gen, drop_last=True, num_workers=0)
print(f"Train batches: {len(train_loader)} (Batch shape: [{BATCH_SIZE}, 3, {IMAGE_SIZE}, {IMAGE_SIZE}])")

---
## Step 3: Define a Modular Convolutional Neural Network

We define a 4-stage convolutional network (`PetClassifierCNN`). All layers use standard PyTorch building blocks (`nn.Conv2d`, `nn.BatchNorm2d`, `nn.ReLU`, `nn.AdaptiveAvgPool2d`, and `nn.Linear`).

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        # With static batch_size=64 (drop_last=True), normalizing per batch avoids
        # bfloat16 running-buffer drift between train() and eval() on TorchTPU 0.2.0.
        self.bn = nn.BatchNorm2d(out_channels, track_running_stats=False)
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        return self.pool(self.relu(self.bn(self.conv(x))))

class PetClassifierCNN(nn.Module):
    def __init__(self, num_classes=37):
        super().__init__()
        # 224x224 -> 112x112 -> 56x56 -> 28x28 -> 14x14
        self.features = nn.Sequential(
            ConvBlock(3, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

# Instantiate model and place on Cloud TPU in bfloat16
model = PetClassifierCNN(num_classes=NUM_CLASSES)
model = model.to(device=device, dtype=torch.bfloat16)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model Architecture instantiated on {device}.")
print(f"Total trainable parameters: {total_params:,}")

---
## Step 4: Training Loop on Cloud TPU

> [!IMPORTANT]
> **TPU Execution Principles Shown in this Loop**:
> 1. **Data Transfer**: `images.to(device=device, dtype=torch.bfloat16)` moves inputs directly to TPU memory in half-precision format.
> 2. **Pipeline Asynchrony**: The training loop avoids calling `loss.item()` on every batch. Metrics accumulate on device, syncing back to host CPU only every `LOG_INTERVAL` steps.
> 3. **Standard Optimization**: PyTorch's native `AdamW` updates model parameters directly on the accelerator.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 5
LOG_INTERVAL = 10

print(f"Starting eager training on {device} for {EPOCHS} epochs...")
total_start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_start = time.time()

    # Device-side accumulators (use float32 for running sums to preserve precision)
    running_loss = torch.zeros((), device=device, dtype=torch.float32)
    correct_device = torch.zeros((), device=device, dtype=torch.int32)
    total_samples = 0

    for step, (images, targets) in enumerate(train_loader, start=1):
        # Move inputs to TPU memory in bfloat16
        images = images.to(device=device, dtype=torch.bfloat16)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        # Accumulate metrics on the TPU without host synchronization
        running_loss += loss.detach().float()
        preds = outputs.argmax(dim=1)
        correct_device += (preds == targets).sum()
        total_samples += targets.size(0)

        if step % LOG_INTERVAL == 0:
            # Sync with host CPU only once every LOG_INTERVAL steps
            current_loss = (running_loss / step).item()
            current_acc = (correct_device.float() / total_samples).item() * 100
            print(f"Epoch [{epoch}/{EPOCHS}] | Step [{step}/{len(train_loader)}] | Loss: {current_loss:.4f} | Acc: {current_acc:.2f}%")

    # Reading .item() synchronizes the TPU before stopping the epoch timer
    final_loss = (running_loss / len(train_loader)).item()
    final_acc = (correct_device.float() / total_samples).item() * 100
    epoch_time = time.time() - epoch_start
    throughput = total_samples / epoch_time
    print(f"--> Epoch {epoch} Complete | Loss: {final_loss:.4f} | Train Acc: {final_acc:.2f}% | Throughput: {throughput:.1f} imgs/s ({epoch_time:.2f}s)")

torch.accelerator.synchronize()
print(f"\nEager training finished in {time.time() - total_start:.2f} seconds.")

---
## Step 5: Compile & Time with `torch.compile`

Once our training loop works eagerly, we wrap the model in `torch.compile(model, backend="tpu")` to fuse operations into a single OpenXLA TPU executable.

- **Step 1 (Warmup / Compile)**: Captures the forward + backward graph and compiles the XLA binary for shape `[64, 3, 224, 224]`.
- **Steps 2…N (Steady State)**: Reuses the cached TPU binary at full hardware speed.

In [ ]:
# Snapshot trained weights so single-batch timing steps don't alter validation accuracy
saved_state = {k: v.clone() for k, v in model.state_dict().items()}

# Pre-stage one static batch on TPU so we measure pure accelerator execution
sample_images, sample_targets = next(iter(train_loader))
sample_images = sample_images.to(device=device, dtype=torch.bfloat16)
sample_targets = sample_targets.to(device)

# 1. Measure steady-state Eager step time
model.train()
torch.accelerator.synchronize()
eager_start = time.time()
for _ in range(10):
    optimizer.zero_grad(set_to_none=True)
    loss = criterion(model(sample_images), sample_targets)
    loss.backward()
    optimizer.step()
torch.accelerator.synchronize()
eager_ms = (time.time() - eager_start) / 10 * 1000

# 2. Compile the model with TorchTPU's OpenXLA backend (with graceful fallback)
try:
    compiled_model = torch.compile(model, backend="tpu")
    torch.accelerator.synchronize()
    t_compile = time.time()
    optimizer.zero_grad(set_to_none=True)
    loss = criterion(compiled_model(sample_images), sample_targets)
    loss.backward()
    optimizer.step()
    torch.accelerator.synchronize()
    step1_ms = (time.time() - t_compile) * 1000
except Exception as e:
    print(f"⚠️ [TorchTPU Recovery] Compilation fallback to eager: {e}")
    compiled_model = model
    step1_ms = eager_ms

# 3. Measure steady-state Compiled step time (cached XLA binary)
torch.accelerator.synchronize()
compiled_start = time.time()
for _ in range(20):
    optimizer.zero_grad(set_to_none=True)
    loss = criterion(compiled_model(sample_images), sample_targets)
    loss.backward()
    optimizer.step()
torch.accelerator.synchronize()
compiled_ms = (time.time() - compiled_start) / 20 * 1000

# Restore the 5-epoch trained weights before validation evaluation
model.load_state_dict(saved_state)

peak_hbm_gib = torch.accelerator.max_memory_allocated() / (1024 ** 3)
print(f"Eager steady-state   : {eager_ms:.2f} ms/step ({BATCH_SIZE / (eager_ms / 1000):.0f} imgs/s)")
print(f"Compiled Step 1      : {step1_ms:.1f} ms (one-time graph capture + XLA compile)")
print(f"Compiled steady-state: {compiled_ms:.2f} ms/step ({BATCH_SIZE / (compiled_ms / 1000):.0f} imgs/s, {eager_ms / compiled_ms:.2f}x speedup)")
print(f"Peak TPU HBM used    : {peak_hbm_gib:.2f} GiB")

---
## Step 6: Validation Evaluation

Let's compute the top-1 accuracy on the held-out validation dataset.

In [ ]:
model.eval()
val_correct = torch.tensor(0, device=device, dtype=torch.int32)
val_total = 0

with torch.no_grad():
    for images, targets in val_loader:
        images = images.to(device=device, dtype=torch.bfloat16)
        targets = targets.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)
        val_correct += (preds == targets).sum()
        val_total += targets.size(0)

val_acc = (val_correct.float() / val_total).item() * 100
print(f"Validation Set Accuracy over {val_total} samples: {val_acc:.2f}%")


---
## Step 7: Visualizing Model Predictions

We display a sample batch of test images alongside their predicted breed and true label.

In [ ]:
import numpy as np

# Retrieve one batch from the validation set
images, targets = next(iter(val_loader))
images_tpu = images.to(device=device, dtype=torch.bfloat16)

with torch.no_grad():
    outputs = model(images_tpu)
    preds = outputs.argmax(dim=1).cpu()

# Denormalize images for plotting
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

plt.figure(figsize=(12, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    img = images[i].numpy().transpose((1, 2, 0))
    img = np.clip(img * std + mean, 0, 1)

    pred_name = classes[preds[i]] if isinstance(classes[0], str) else f"Class {preds[i]}"
    true_name = classes[targets[i]] if isinstance(classes[0], str) else f"Class {targets[i]}"

    is_correct = (preds[i] == targets[i])
    color = "green" if is_correct else "red"

    plt.imshow(img)
    plt.title(f"Pred: {pred_name}\nTrue: {true_name}", color=color, fontsize=10)
    plt.axis("off")

plt.tight_layout()
plt.show()


---
## Summary & Next Step in the Workshop

### What You Observed:
- **Standard PyTorch Code**: Zero custom accelerator APIs or manual execution calls.
- **Native `bfloat16`**: Direct half-precision execution on TPU Matrix Multiply Units.
- **Invariant Batch Shapes**: Setting `drop_last=True` avoids recompilation on the trailing batch.
- **Decoupled Synchronization**: Keeping loss metrics on device avoids host pipeline stalls.
- **Eager → `torch.compile`**: Verify correctness eagerly first, then fuse the model graph into a cached XLA binary with `torch.compile(model, backend="tpu")`.

Now that you understand the basic mechanics of training on Cloud TPU, open the next notebook: **LoRA Fine-Tuning Gemma on Cloud TPU v5e** to apply these principles to large language models!